In [1]:
from pathlib import Path
import os
import time
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_from_disk
from transformers import pipeline

os.environ["TOKENIZERS_PARALLELISM"] = "false"

PROJECT_ROOT = Path("/home/ridwan/projects/afrispeech-project")

CLINICAL_ARROW_PATH = PROJECT_ROOT / "data" / "processed" / "clinical_test_arrow"
GENERAL_ARROW_PATH = PROJECT_ROOT / "data" / "processed" / "general_test_arrow"

PREDICTIONS_DIR = PROJECT_ROOT / "results" / "predictions"
LOGS_DIR = PROJECT_ROOT / "logs"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Clinical path exists:", CLINICAL_ARROW_PATH.exists())
print("General path exists:", GENERAL_ARROW_PATH.exists())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: /home/ridwan/projects/afrispeech-project
Clinical path exists: True
General path exists: True
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


In [2]:
clinical_ds = load_from_disk(str(CLINICAL_ARROW_PATH))
general_ds = load_from_disk(str(GENERAL_ARROW_PATH))

print("Clinical dataset:", clinical_ds)
print("General dataset:", general_ds)

print("Clinical samples:", len(clinical_ds))
print("General samples:", len(general_ds))

print("Columns:", clinical_ds.column_names)

Clinical dataset: Dataset({
    features: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration'],
    num_rows: 3606
})
General dataset: Dataset({
    features: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration'],
    num_rows: 2712
})
Clinical samples: 3606
General samples: 2712
Columns: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration']


In [3]:
MODEL_NAME = "openai/whisper-small"
MODEL_SHORT_NAME = "whisper_small"

device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_NAME,
    torch_dtype=torch_dtype,
    device=device,
)

print("Loaded model:", MODEL_NAME)
print("Device:", "cuda" if device == 0 else "cpu")

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Loaded model: openai/whisper-small
Device: cuda


In [8]:
sample = clinical_ds[1]

print("Audio ID:", sample["audio_id"])
print("Domain:", sample["domain"])
print("Accent:", sample["accent"])
print("Country:", sample["country"])
print("Duration:", sample["duration"])
print("Reference:", sample["transcript"])

audio_input = {
    "array": sample["audio"]["array"],
    "sampling_rate": sample["audio"]["sampling_rate"]
}

prediction = asr_pipe(
    audio_input,
    generate_kwargs={
        "language": "english",
        "task": "transcribe"
    }
)

print("\nPrediction:")
print(prediction["text"])

Audio ID: 80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b9160663e9c7d36d3b275120
Domain: clinical
Accent: setswana
Country: ZA
Duration: 4.526984214782715
Reference: Ask the patient about allergies totape and skin antiseptics.


Prediction:
 Ask the patients about allergies to tape and skin antiseptics.


In [9]:
def run_whisper_inference(
    dataset,
    output_csv_path,
    model_name,
    model_short_name,
    domain_name,
    max_samples=None,
    flush_every=25
):
    """
    Run Whisper inference on a Hugging Face Arrow dataset and save predictions to CSV.
    """

    output_csv_path = Path(output_csv_path)

    n_samples = len(dataset) if max_samples is None else min(max_samples, len(dataset))

    rows = []
    start_time = time.time()

    for i in tqdm(range(n_samples), desc=f"Running {model_short_name} on {domain_name}"):
        sample = dataset[i]

        audio_input = {
            "array": sample["audio"]["array"],
            "sampling_rate": sample["audio"]["sampling_rate"]
        }

        try:
            pred = asr_pipe(
                audio_input,
                generate_kwargs={
                    "language": "english",
                    "task": "transcribe"
                }
            )

            prediction_text = pred["text"]

            error_message = ""

        except Exception as e:
            prediction_text = ""
            error_message = str(e)

        rows.append({
            "row_index": i,
            "audio_id": sample.get("audio_id", ""),
            "speaker_id": sample.get("speaker_id", ""),
            "domain": sample.get("domain", ""),
            "accent": sample.get("accent", ""),
            "country": sample.get("country", ""),
            "gender": sample.get("gender", ""),
            "age_group": sample.get("age_group", ""),
            "duration": sample.get("duration", ""),
            "model_name": model_name,
            "model_short_name": model_short_name,
            "reference": sample.get("transcript", ""),
            "prediction": prediction_text,
            "error": error_message
        })

        # Save partial progress
        if (i + 1) % flush_every == 0:
            pd.DataFrame(rows).to_csv(output_csv_path, index=False)

    # Final save
    df = pd.DataFrame(rows)
    df.to_csv(output_csv_path, index=False)

    elapsed_minutes = round((time.time() - start_time) / 60, 2)

    print("Saved predictions to:", output_csv_path)
    print("Samples processed:", len(df))
    print("Elapsed time minutes:", elapsed_minutes)

    return df

In [10]:
test_output_path = PREDICTIONS_DIR / "test_whisper_small_clinical_5samples.csv"

test_predictions_df = run_whisper_inference(
    dataset=clinical_ds,
    output_csv_path=test_output_path,
    model_name=MODEL_NAME,
    model_short_name=MODEL_SHORT_NAME,
    domain_name="clinical",
    max_samples=5,
    flush_every=1
)

display(test_predictions_df[["audio_id", "accent", "reference", "prediction", "error"]])

Running whisper_small on clinical:   0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /home/ridwan/projects/afrispeech-project/results/predictions/test_whisper_small_clinical_5samples.csv
Samples processed: 5
Elapsed time minutes: 0.01


,audio_id,accent,reference,prediction,error
0,27a83595-3d3f-4a6b-b909-7b8f364d736b/7cb8dfb31...,setswana,Protection of the host immune mechanism mighti...,Protection of the host immune mechanism might...,
1,80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b...,setswana,Ask the patient about allergies totape and ski...,Ask the patients about allergies to tape and ...,
2,13f2d282-bda1-4bc3-83b8-7a0d06e5eda0/933137f33...,setswana,"The short, mucus-secreting cells between the c...",The short macro secreting cells between the c...,
3,99f6019b-4505-4c60-a462-79b0eb6e689a/2cb25e2dd...,setswana,Aspiration is terminated when aspirated materi...,6. Aspiration is terminated when aspirated ma...,
4,13600ba0-a4d5-46f7-9235-776cd60bfdee/1404f622c...,setswana,Identify the appropriate landmarks for the sit...,Identify the appropriate landmarks for the si...,


In [11]:
clinical_output_path = PREDICTIONS_DIR / "whisper_small_clinical_predictions.csv"

clinical_predictions_df = run_whisper_inference(
    dataset=clinical_ds,
    output_csv_path=clinical_output_path,
    model_name=MODEL_NAME,
    model_short_name=MODEL_SHORT_NAME,
    domain_name="clinical",
    max_samples=None,
    flush_every=25
)

display(clinical_predictions_df.head())

Running whisper_small on clinical:   0%|          | 0/3606 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved predictions to: /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
Samples processed: 3606
Elapsed time minutes: 8.05


,row_index,audio_id,speaker_id,domain,accent,country,gender,age_group,duration,model_name,model_short_name,reference,prediction,error
0,0,27a83595-3d3f-4a6b-b909-7b8f364d736b/7cb8dfb31...,cdf91cf6e59ee411b985a40a955d4d1f,clinical,setswana,BW,Female,26-40,9.730000,openai/whisper-small,whisper_small,Protection of the host immune mechanism mighti...,Protection of the host immune mechanism might...,
1,1,80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b...,6d53c4d9134a175a75a020155220ca7f,clinical,setswana,ZA,Female,19-25,4.526984,openai/whisper-small,whisper_small,Ask the patient about allergies totape and ski...,Ask the patients about allergies to tape and ...,
2,2,13f2d282-bda1-4bc3-83b8-7a0d06e5eda0/933137f33...,6d53c4d9134a175a75a020155220ca7f,clinical,setswana,ZA,Female,19-25,8.563991,openai/whisper-small,whisper_small,"The short, mucus-secreting cells between the c...",The short macro secreting cells between the c...,
3,3,99f6019b-4505-4c60-a462-79b0eb6e689a/2cb25e2dd...,38a2628beaa524416a8daf826e9d10e2,clinical,setswana,BW,Female,26-40,9.773991,openai/whisper-small,whisper_small,Aspiration is terminated when aspirated materi...,6. Aspiration is terminated when aspirated ma...,
4,4,13600ba0-a4d5-46f7-9235-776cd60bfdee/1404f622c...,1e4033e1ce6f7cce51ca05b1908fc14a,clinical,setswana,BW,Female,19-25,6.258980,openai/whisper-small,whisper_small,Identify the appropriate landmarks for the sit...,Identify the appropriate landmarks for the si...,


In [13]:
clinical_saved = pd.read_csv(PREDICTIONS_DIR / "whisper_small_clinical_predictions.csv")
# general_saved = pd.read_csv(PREDICTIONS_DIR / "whisper_small_general_predictions.csv")

print("Clinical predictions:", clinical_saved.shape)
# print("General predictions:", general_saved.shape)

display(clinical_saved.head())
# display(general_saved.head())

Clinical predictions: (3606, 14)


,row_index,audio_id,speaker_id,domain,accent,country,gender,age_group,duration,model_name,model_short_name,reference,prediction,error
0,0,27a83595-3d3f-4a6b-b909-7b8f364d736b/7cb8dfb31...,cdf91cf6e59ee411b985a40a955d4d1f,clinical,setswana,BW,Female,26-40,9.730000,openai/whisper-small,whisper_small,Protection of the host immune mechanism mighti...,Protection of the host immune mechanism might...,NaN
1,1,80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b...,6d53c4d9134a175a75a020155220ca7f,clinical,setswana,ZA,Female,19-25,4.526984,openai/whisper-small,whisper_small,Ask the patient about allergies totape and ski...,Ask the patients about allergies to tape and ...,NaN
2,2,13f2d282-bda1-4bc3-83b8-7a0d06e5eda0/933137f33...,6d53c4d9134a175a75a020155220ca7f,clinical,setswana,ZA,Female,19-25,8.563991,openai/whisper-small,whisper_small,"The short, mucus-secreting cells between the c...",The short macro secreting cells between the c...,NaN
3,3,99f6019b-4505-4c60-a462-79b0eb6e689a/2cb25e2dd...,38a2628beaa524416a8daf826e9d10e2,clinical,setswana,BW,Female,26-40,9.773991,openai/whisper-small,whisper_small,Aspiration is terminated when aspirated materi...,6. Aspiration is terminated when aspirated ma...,NaN
4,4,13600ba0-a4d5-46f7-9235-776cd60bfdee/1404f622c...,1e4033e1ce6f7cce51ca05b1908fc14a,clinical,setswana,BW,Female,19-25,6.258980,openai/whisper-small,whisper_small,Identify the appropriate landmarks for the sit...,Identify the appropriate landmarks for the si...,NaN
